# Agent Offline Evaluation — Agent Evaluators

Uses `AIAgentConverter` to transform captured agent threads into evaluation SDK format, then runs:

- **IntentResolutionEvaluator** — did the agent correctly identify user intent?
- **ToolCallAccuracyEvaluator** — did the agent call the right tools?
- **TaskAdherenceEvaluator** — did the response adhere to the agent's system prompt?

> **⚠️ SDK incompatibility (Spring 2026)**
>
> As of `azure-ai-projects` 2.1.0 / `azure-ai-evaluation` 1.16.5, `AIAgentConverter.convert(thread_id, run_id)` fails on this SDK combination — the `FDPAgentDataRetriever` path it dispatches to calls `project_client.agents.runs.list(...)`, which doesn't exist on `AIProjectClient` 2.1.0. The converter cell below is wrapped in try/except so the notebook still runs; downstream cells skip cleanly when conversion fails. When Microsoft ships a fix or a Responses-API-aware converter, the cell will produce eval-ready data again — the `thread_id` and `run_id` columns in `test_data.jsonl` are ready for that future step.


In [1]:
import hashlib
import json
import os
import subprocess
from pathlib import Path

from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.evaluation import (
    AIAgentConverter,
    IntentResolutionEvaluator,
    ToolCallAccuracyEvaluator,
    TaskAdherenceEvaluator,
    AzureOpenAIModelConfiguration,
    evaluate,
)
from dotenv import load_dotenv

## Environment

In [2]:
repo_root = Path(subprocess.run(
    'git rev-parse --show-toplevel', shell=True, capture_output=True, text=True
).stdout.strip())
load_dotenv(repo_root / '.env', override=True)

CHAT_MODEL = os.environ['CHAT_MODEL']

SUB_ID = subprocess.run(
    'az account show --query id -o tsv', shell=True, capture_output=True, text=True
).stdout.strip()
SUFFIX = hashlib.sha256((SUB_ID + 'v2').encode()).hexdigest()[:6]
PROJECT_ENDPOINT = f'https://aif-core-{SUFFIX}.services.ai.azure.com/api/projects/project-admin-{SUFFIX}'
AOAI_ENDPOINT    = f'https://aif-core-{SUFFIX}.services.ai.azure.com/'

In [3]:
credential = DefaultAzureCredential()

project_client = AIProjectClient(
    endpoint=PROJECT_ENDPOINT,
    credential=credential,
)

# Python 3.13 + azure-ai-evaluation 1.16.x workaround: omit `credential` from
# AzureOpenAIModelConfiguration (SDK validates via isinstance(value, Any), which
# Python 3.13 made into a hard TypeError). Pass credential as a kwarg to each
# evaluator below instead. See 08-06-00 README for details.
model_config = AzureOpenAIModelConfiguration(
    azure_endpoint=AOAI_ENDPOINT,
    azure_deployment=CHAT_MODEL,
)


## Load test data (thread/run IDs from 08-05-01)

In [4]:
lab_dir        = repo_root / '08-agents' / '08-06-agent-offline-evaluation'
test_data_path = lab_dir / 'test_data.jsonl'

test_records = []
with open(test_data_path) as f:
    for line in f:
        line = line.strip()
        if line:
            test_records.append(json.loads(line))

if not test_records:
    raise RuntimeError('test_data.jsonl is empty. Run 08-05-01 first.')

print(f'Loaded {len(test_records)} records')
for r in test_records:
    print(f'  thread={r.get("thread_id", "MISSING")}  run={r.get("run_id", "MISSING")}')

Loaded 5 records
  thread=thread_h3J5ve7MOUN0hIrMGeFIp37K  run=run_KaivezoDJWoeb17CN5NNYZYz
  thread=thread_PlCqO18aqnSTrdXiLyAF0eAv  run=run_8SrpWj3fdS0CfyhS5o5oeJb0
  thread=thread_B3Zrwu0T61bTLq3IVXuDl9nP  run=run_IvXfYmwLpbzzyJ4LehLaEP5S
  thread=thread_zVpIyv2acfAJskUpiZJpTc4i  run=run_61YauBDe9bCKRPaYY0Da6Mmm
  thread=thread_PNtjlbFt6lcIc4NnyB8B1plf  run=run_ZxrS7qYKN1vGrXLmhPop77ai


## Convert agent thread using AIAgentConverter

`AIAgentConverter.convert()` fetches the full thread history (messages, tool calls, tool results) and transforms it into the JSONL format the evaluation SDK expects.

In [5]:
# Try to convert the first captured thread/run via AIAgentConverter.
# See the warning in the intro markdown — the converter is upstream-blocked on
# this SDK version, so we catch the AttributeError and let the notebook continue.
agent_eval_data = None
converter_error = None

try:
    converter = AIAgentConverter(project_client=project_client)
    first_record = test_records[0]
    thread_id = first_record['thread_id']
    run_id    = first_record['run_id']
    print(f'Converting thread_id={thread_id}, run_id={run_id}')
    agent_eval_data = converter.convert(thread_id=thread_id, run_id=run_id)
    print(f'✓ Converted {len(agent_eval_data)} records')
    print(json.dumps(agent_eval_data[0] if agent_eval_data else {}, indent=2))
except AttributeError as e:
    converter_error = str(e)
    print('⚠️  AIAgentConverter is blocked on this SDK version:')
    print(f'    {type(e).__name__}: {e}')
    print()
    print('    This is a known upstream issue with azure-ai-projects 2.1.0 /')
    print('    azure-ai-evaluation 1.16.5. Downstream cells will skip cleanly.')


Class AIAgentConverter: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class FDPAgentDataRetriever: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class AIAgentDataRetriever: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.


Converting thread_id=thread_h3J5ve7MOUN0hIrMGeFIp37K, run_id=run_KaivezoDJWoeb17CN5NNYZYz
⚠️  AIAgentConverter is blocked on this SDK version:
    AttributeError: 'AgentsOperations' object has no attribute 'runs'

    This is a known upstream issue with azure-ai-projects 2.1.0 /
    azure-ai-evaluation 1.16.5. Downstream cells will skip cleanly.


## Save converted data

In [6]:
agent_eval_path = lab_dir / 'agent_eval_data.jsonl'
if agent_eval_data:
    with open(agent_eval_path, 'w') as f:
        for record in agent_eval_data:
            f.write(json.dumps(record) + '\n')
    print(f'Saved to {agent_eval_path}')
else:
    print(f'Skipped — converter is blocked, no records to save.')
    print(f'(Would have written to {agent_eval_path}.)')


Skipped — converter is blocked, no records to save.
(Would have written to /home/jp/development/corticalstack/foundry-nextgen/08-agents/08-06-agent-offline-evaluation/agent_eval_data.jsonl.)


## Run agent-specific evaluators

In [7]:
if agent_eval_data:
    agent_results = evaluate(
        data=str(agent_eval_path),
        evaluators={
            'intent_resolution':  IntentResolutionEvaluator(model_config=model_config, credential=credential),
            'tool_call_accuracy': ToolCallAccuracyEvaluator(model_config=model_config, credential=credential),
            'task_adherence':     TaskAdherenceEvaluator(model_config=model_config, credential=credential),
        },
    )
    print('Agent evaluator metrics:', agent_results.get('metrics', {}))
else:
    agent_results = {'metrics': {}, 'rows': []}
    print('Skipped — see converter error above.')
    print('When the SDK ships a fix, re-run from the converter cell.')


Skipped — see converter error above.
When the SDK ships a fix, re-run from the converter cell.


## Display agent evaluation results

In [8]:
import sys
sys.path.insert(0, str(lab_dir))
from evaluation_helpers import display_agent_eval_results

display_agent_eval_results(agent_results)

### Agent Evaluation Results

*No agent evaluator metrics found in results.*

## What each evaluator measures

| Evaluator | What it checks | Score range |
|---|---|---|
| **IntentResolutionEvaluator** | Whether the agent correctly identified user intent | 1–5 |
| **ToolCallAccuracyEvaluator** | Whether the agent called the right tools with correct arguments | 1–5 |
| **TaskAdherenceEvaluator** | Whether the response follows the agent's system prompt and task | 1–5 |